# F1 AI Race Strategist — Transformer Training (Colab GPU)

Trains the TabTransformer lap time predictor on GPU.
Expected time: ~15 min on Colab T4.

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Upload your parquet files or mount Google Drive
3. Run all cells in order


In [ ]:
# ── 1. Clone repo and install dependencies ──────────────────────────────────
import subprocess
result = subprocess.run(
    ['git', 'clone', 'https://github.com/Shreyansh262/f1-strategist.git'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
%cd f1-strategist

In [ ]:
!pip install fastf1 pandera mlflow shap -q

In [ ]:
# ── 2. Verify GPU ────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# ── 3. Mount Drive and copy parquet files ───────────────────────────────────
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

# Adjust this path to where you stored your parquet files in Drive
DRIVE_PARQUET_DIR = Path('/content/drive/MyDrive/f1-strategist/data/raw')
LOCAL_PARQUET_DIR = Path('data/raw')
LOCAL_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_PARQUET_DIR.exists():
    files = list(DRIVE_PARQUET_DIR.glob('*.parquet'))
    for f in files:
        shutil.copy(f, LOCAL_PARQUET_DIR / f.name)
    print(f'Copied {len(files)} parquet files')
else:
    print('Drive path not found — adjust DRIVE_PARQUET_DIR above')

In [ ]:
# ── 4. Verify data ───────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

import pandas as pd
files = list(LOCAL_PARQUET_DIR.glob('*.parquet'))
print(f'{len(files)} parquet files found')

dfs = [pd.read_parquet(f) for f in sorted(files)]
raw = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(raw)}')
print('Seasons:', sorted(raw['Season'].unique()))

In [ ]:
# ── 5. Copy circuit_map.json if it exists in Drive ──────────────────────────
DRIVE_MAPPINGS = Path('/content/drive/MyDrive/f1-strategist/data/mappings')
LOCAL_MAPPINGS = Path('data/mappings')
LOCAL_MAPPINGS.mkdir(parents=True, exist_ok=True)

if (DRIVE_MAPPINGS / 'circuit_map.json').exists():
    shutil.copy(DRIVE_MAPPINGS / 'circuit_map.json', LOCAL_MAPPINGS / 'circuit_map.json')
    print('circuit_map.json copied from Drive')
else:
    print('No circuit_map.json found — will be created during training')

In [ ]:
# ── 6. Train the transformer ────────────────────────────────────────────────
from src.models.lap_time.train_transformer import train
train()

In [ ]:
# ── 7. Copy model back to Drive ─────────────────────────────────────────────
DRIVE_MODELS = Path('/content/drive/MyDrive/f1-strategist/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

for fname in ['transformer_lap.pt', 'scaler_transformer.joblib']:
    src = Path('models') / fname
    if src.exists():
        shutil.copy(src, DRIVE_MODELS / fname)
        print(f'Saved {fname} → Drive')

# Also save circuit_map.json
circuit_map = Path('data/mappings/circuit_map.json')
if circuit_map.exists():
    shutil.copy(circuit_map, DRIVE_MAPPINGS / 'circuit_map.json')
    print('circuit_map.json → Drive')

# Save reports
DRIVE_REPORTS = Path('/content/drive/MyDrive/f1-strategist/reports')
if Path('reports').exists():
    shutil.copytree('reports', str(DRIVE_REPORTS), dirs_exist_ok=True)
    print('Reports → Drive')

In [ ]:
# ── 8. Quick sanity check ───────────────────────────────────────────────────
import torch, joblib
from src.models.lap_time.train_transformer import TabTransformer, predict_with_uncertainty, CAT_CARDINALITIES, CONT_FEATURE_INDICES

checkpoint = torch.load('models/transformer_lap.pt', map_location='cpu')
cfg = checkpoint['model_config']
model = TabTransformer(**cfg)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Best val MAE from checkpoint:', checkpoint['best_val_mae'])
print('Features:', checkpoint['feature_columns'])

# Test inference
x_cat  = torch.tensor([[5, 1, 0]], dtype=torch.long)   # Bahrain, Medium, Era0
x_cont = torch.zeros(1, len(CONT_FEATURE_INDICES))
mean_p, std_p = predict_with_uncertainty(model, x_cat, x_cont, n_samples=30)
print(f'Sample prediction: {mean_p[0]:.2f}s ± {std_p[0]:.2f}s')